# Predictive Customer Analytics

End-to-end analysis of a 2,000-customer retail dataset completed as part of the LSE Data Analytics Career Accelerator. The project uses Python and R to explore customer behaviour, model loyalty-point accumulation, identify actionable customer segments, and analyse review sentiment.

**Methods:** data cleaning and validation, linear regression, decision-tree regression, k-means clustering, NLP/sentiment analysis, statistical diagnostics and business interpretation.

> The source dataset is not included in this repository. The notebook is presented as a portfolio example of the analytical workflow and code.


## 1. Load and explore the data


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm 
from statsmodels.formula.api import ols


In [ ]:
# Load the CSV file as reviews
reviews = pd.read_csv("turtle_reviews.csv")

# View the DataFrame
display(reviews.head())

# View the shape, to ensure appropriateness 
print("Shape:", reviews.shape)
reviews.info()


In [ ]:
# Any missing values?
missing_values = reviews.isna().sum()

print("Missing values per column:")
print(missing_values)

print("\nTotal missing values in DataFrame:", reviews.isna().sum().sum())
# There are no missing values.


In [ ]:
# Basic descriptive statistics.
display(reviews.describe())


In [ ]:
# Basic descriptive statistics including categorical columns
display(reviews.describe(include="all"))


## 2. Drop columns


In [ ]:
# Drop unnecessary columns
reviews = reviews.drop(columns=["language", "platform", "review", "summary"])

# View column names
print("Remaining columns:")
print(reviews.columns.tolist())


## 3. Rename columns


In [ ]:
# Rename the column headers
reviews = reviews.rename(columns={
    "remuneration (k£)": "remuneration",
    "spending_score (1-100)": "spending_score"
})

# View column names
print("Updated column names:")
print(reviews.columns.tolist())


## 4. Save the DataFrame as a CSV file


In [ ]:
# Create a CSV file as output
reviews.to_csv("turtle_reviews_clean.csv", index=False)

print("Clean dataset saved as turtle_reviews_clean.csv")


In [ ]:
# Import new CSV file with Pandas
reviews_clean = pd.read_csv("turtle_reviews_clean.csv")

# View DataFrame
display(reviews_clean.head())

# View the shape, to ensure appropriateness
print("Shape:", reviews_clean.shape)
reviews_clean.info()


## 5. Linear regression


### 5a) spending vs loyalty


In [ ]:
# 5a) Spending vs Loyalty

# Define independent variable (X)
X = reviews_clean["spending_score"]

# Define dependent variable (y)
y = reviews_clean["loyalty_points"]

# Add constant (intercept) to the model
X_const = sm.add_constant(X)

# Create OLS regression model
model_spending = sm.OLS(y, X_const).fit()

# Print summary of regression metrics
print(model_spending.summary())


In [ ]:
# Extract the estimated parameters (coefficients)
estimated_parameters = model_spending.params
print("Estimated parameters:")
print(estimated_parameters)

# Extract the standard errors
standard_errors = model_spending.bse
print("\nStandard errors:")
print(standard_errors)

# Extract the predicted values
predicted_values = model_spending.predict(X_const)
print("\nFirst 5 predicted values:")
print(predicted_values.head())


In [ ]:
# Set the X coefficient and the constant to generate the regression table
regression_table = pd.DataFrame({
    "Variable": ["Constant", "Spending Score"],
    "Coefficient": [
        model_spending.params["const"],
        model_spending.params["spending_score"]
    ],
    "Standard Error": [
        model_spending.bse["const"],
        model_spending.bse["spending_score"]
    ]
})

# View the output
display(regression_table)


In [ ]:
 # Plot the graph with a regression line
plt.figure(figsize=(7, 5))

# Scatter plot of observed data
plt.scatter(
    reviews_clean["spending_score"],
    reviews_clean["loyalty_points"],
    alpha=0.5
)

# Regression line
plt.plot(
    reviews_clean["spending_score"],
    model_spending.predict(X_const),
    color="red"
)

# Labels and title
plt.xlabel("Spending Score")
plt.ylabel("Loyalty Points")
plt.title("Loyalty Points vs Spending Score")

plt.show()


**Summary of 5a) spending vs loyalty:**

- The linear regression analysis demonstrates a strong positive relationship between spending score and loyalty point accumulation.
- The model explains 45.2% of the variation in loyalty points (R² = 0.452), indicating substantial explanatory power for a single behavioural variable.
- The spending score coefficient (β = 33.06, p < 0.001) indicates that each one-point increase in spending score is associated with an increase of approximately 33 loyalty points, on average.
- The estimate is highly precise, with a small standard error (0.81) and a narrow confidence interval, indicating a stable and reliable relationship.
- The intercept is not statistically significant and has limited practical meaning, as a spending score of zero is not behaviourally realistic.
- Visual inspection of the regression plot shows that loyalty points are closely clustered around the regression line up to a spending score of approximately 50, suggesting a predictable relationship for low-to-mid spenders.
- Beyond this level, the dispersion of loyalty points increases, with observations spread both above and below the regression line.
- This widening spread indicates greater heterogeneity in loyalty behaviour among higher-spending customers, likely driven by additional factors such as promotional exposure, reward thresholds, or purchase frequency.
- Overall, spending score is the strongest single predictor of loyalty engagement among the variables tested.
- From a marketing perspective, loyalty programmes should prioritise behaviour-based segmentation, using spending score to identify high-potential customers and design targeted incentives that encourage progression to higher spending tiers.


### 5b) Remuneration vs loyalty


In [ ]:
# Define independent variable (X)
X = reviews_clean["remuneration"]

# Define dependent variable (y)
y = reviews_clean["loyalty_points"]

# Add constant (intercept)
X_const = sm.add_constant(X)

# Create OLS regression model
model_remuneration = sm.OLS(y, X_const).fit()

# Print summary of regression metrics
print(model_remuneration.summary())


In [ ]:
# Extract the estimated parameters (coefficients)
estimated_parameters_rem = model_remuneration.params
print("Estimated parameters:")
print(estimated_parameters_rem)

# Extract the standard errors
standard_errors_rem = model_remuneration.bse
print("\nStandard errors:")
print(standard_errors_rem)

# Extract the predicted values
predicted_values_rem = model_remuneration.predict(X_const)
print("\nFirst 5 predicted values:")
print(predicted_values_rem.head())


In [ ]:
# Set the X coefficient and the constant to generate the regression table
regression_table_rem = pd.DataFrame({
    "Variable": ["Constant", "Remuneration"],
    "Coefficient": [
        model_remuneration.params["const"],
        model_remuneration.params["remuneration"]
    ],
    "Standard Error": [
        model_remuneration.bse["const"],
        model_remuneration.bse["remuneration"]
    ]
})

# View the output
display(regression_table_rem)


In [ ]:
# Plot graph with regression line
plt.figure(figsize=(7, 5))

# Scatter plot
plt.scatter(
    reviews_clean["remuneration"],
    reviews_clean["loyalty_points"],
    alpha=0.5
)

# Regression line
plt.plot(
    reviews_clean["remuneration"],
    model_remuneration.predict(X_const),
    color="red"
)

# Labels and title
plt.xlabel("Remuneration (k£)")
plt.ylabel("Loyalty Points")
plt.title("Loyalty Points vs Remuneration")

plt.show()


**Summary of 5b) remuneration vs loyalty:**
- The linear regression analysis identifies a statistically significant positive relationship between customer remuneration and loyalty point accumulation.
- The model explains 38.0% of the variance in loyalty points (R² = 0.380), indicating a moderately strong but incomplete explanatory relationship.
- The remuneration coefficient (β = 34.19, p < 0.001) suggests that for each additional £1,000 of income, customers earn approximately 34 additional loyalty points, on average.
- The estimated coefficient is precise, with a small standard error (0.98), supporting the reliability of the relationship.
- The constant term is not statistically significant and has limited practical interpretation.
- Visual inspection of the regression plot shows that loyalty points are tightly clustered around the regression line up to approximately £55k, indicating a more predictable relationship in this income range.
- Beyond £55k, loyalty point outcomes become increasingly dispersed, with observations spread both above and below the regression line.
- This widening variance suggests that income alone is insufficient to explain loyalty behaviour among higher-income customers.
- When compared with spending score (Exercise 5a), remuneration is a weaker predictor of loyalty engagement, reinforcing the importance of behavioural over demographic variables.
- From a marketing perspective, remuneration may be useful for baseline segmentation, but effective loyalty strategies—particularly for high-income customers—should prioritise spending behaviour, engagement patterns, and personalised incentives rather than income alone.


### 5c) age vs loyalty


In [ ]:
# Define independent variable (X)
X = reviews_clean["age"]

# Define dependent variable (y)
y = reviews_clean["loyalty_points"]

# Add constant (intercept)
X_const = sm.add_constant(X)

# Create OLS regression model
model_age = sm.OLS(y, X_const).fit()

# Print summary of regression metrics
print(model_age.summary())


In [ ]:
# Extract the estimated parameters (coefficients)
estimated_parameters_age = model_age.params
print("Estimated parameters:")
print(estimated_parameters_age)

# Extract the standard errors
standard_errors_age = model_age.bse
print("\nStandard errors:")
print(standard_errors_age)

# Extract the predicted values
predicted_values_age = model_age.predict(X_const)
print("\nFirst 5 predicted values:")
print(predicted_values_age.head())


In [ ]:
# Set the X coefficient and the constant to generate the regression table
regression_table_age = pd.DataFrame({
    "Variable": ["Constant", "Age"],
    "Coefficient": [
        model_age.params["const"],
        model_age.params["age"]
    ],
    "Standard Error": [
        model_age.bse["const"],
        model_age.bse["age"]
    ]
})

# View the output
display(regression_table_age)


In [ ]:
# Plot graph with regression line
plt.figure(figsize=(7, 5))

# Scatter plot
plt.scatter(
    reviews_clean["age"],
    reviews_clean["loyalty_points"],
    alpha=0.5
)

# Regression line
plt.plot(
    reviews_clean["age"],
    model_age.predict(X_const),
    color="red"
)

# Labels and title
plt.xlabel("Age")
plt.ylabel("Loyalty Points")
plt.title("Loyalty Points vs Age")

plt.show()


**Summary of 5c) age vs loyalty:**
- The linear regression analysis shows no meaningful relationship between customer age and loyalty point accumulation.
- The model explains only 0.2% of the variation in loyalty points (R² = 0.002), indicating negligible explanatory power.
- The age coefficient (β = −4.01) suggests a very small negative association between age and loyalty points.
- This relationship is not statistically significant (p ≈ 0.058), and the confidence interval includes zero.
- The standard error (2.11) is large relative to the coefficient, indicating a high level of uncertainty in the estimate.
- Visual inspection of the regression plot shows a flat to slightly negative regression line, confirming the lack of a linear relationship.
- Loyalty points are widely dispersed across all age groups, with no clear clustering around the regression line.
- Although some observations lie above the regression line—particularly between ages 30 and 40—this does not form a consistent or reliable pattern.
- Predicted loyalty values remain largely flat across age, reinforcing the weak predictive value of age.
- From a marketing perspective, age-based segmentation is unlikely to be effective for predicting or influencing loyalty engagement.


## 6. Observations and insights


**Spending Score vs Loyalty Points (5a)**

- A strong positive linear relationship was observed between spending score and loyalty points (R² = 0.452).
- Spending score is the most powerful single predictor of loyalty engagement, with each one-point increase associated with ~33 additional loyalty points.
- The relationship is stable for low-to-mid spenders but becomes more variable at higher spending levels, suggesting additional behavioural drivers.
- Suggestion: Prioritise behaviour-based segmentation and design targeted loyalty incentives that encourage medium spenders to increase engagement.

**Remuneration vs Loyalty Points (5b)**

- Remuneration shows a statistically significant but weaker relationship with loyalty points (R² = 0.380).
- Income explains some variation in loyalty behaviour, particularly at lower to mid income levels, but predictive power decreases for higher earners.
- Suggestion: Use remuneration only as a secondary segmentation variable and avoid assuming loyalty engagement based on income alone.

**Age vs Loyalty Points (5c)**

- Age shows no meaningful relationship with loyalty engagement (R² = 0.002).
- The regression line is effectively flat, and age is not a statistically significant predictor.
- Suggestion: Avoid age-based targeting for loyalty programmes.


The analysis examined the relationship between loyalty points and three potential predictors: spending score, remuneration, and age. The results show a clear hierarchy of drivers. Spending score is the strongest predictor of loyalty engagement, explaining over 45% of the variation in loyalty points. This confirms that behavioural measures are far more informative than demographic characteristics when predicting customer loyalty outcomes. Remuneration also demonstrates a positive and statistically significant relationship with loyalty points; however, its explanatory power is weaker than spending behaviour and becomes less reliable at higher income levels. In contrast, age has virtually no explanatory value and does not meaningfully influence loyalty engagement.

From a business perspective, these findings suggest that Turtle Games should adopt a behaviour-led loyalty strategy. Marketing resources should prioritise customers based on spending patterns rather than age or income, using targeted incentives to encourage progression among medium spenders and personalised rewards for high-value customers. Future analysis could explore multivariate models combining spending score and remuneration, non-linear relationships, or the impact of promotional campaigns on loyalty accumulation. Incorporating additional behavioural variables would further improve predictive accuracy and support more effective loyalty programme design.


# 2. Decision-tree regression


## 1. Load and prepare the data


In [ ]:
# Import all the necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Settings for the notebook
warnings.filterwarnings("ignore")
plt.rcParams['figure.figsize'] = [15, 10]


In [ ]:
# Import the cleaned CSV from Week 1
df = pd.read_csv("turtle_reviews_clean.csv")

# Sense-check
df.head(), df.info()


In [ ]:
# Create your new DataFrame
df_model = df[["age", "remuneration", "spending_score", "loyalty_points"]].copy()

# Explore the new DataFrame
df_model.head(), df_model.info(), df_model.describe()



In [ ]:
# Specify Y (dependent variable)
y = df_model["loyalty_points"]

# Specify X (independent variables)
X = df_model.drop(columns="loyalty_points")

# Sense-check
X.head(), y.head(), X.shape, y.shape


In [ ]:
# Review X
print("X (independent variables):")
display(X.head())
print("\nX shape:", X.shape)
print("\nX info:")
X.info()

# Review y
print("\ny (dependent variable):")
display(y.head())
print("\ny shape:", y.shape)


## 2. Create train and test data sets.


In [ ]:
# Split the data into train and test sets (70:30)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

# Sense-check the split
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


## 3. Create Decision tree regressor


In [ ]:
# Create your decision tree regressor
regressor = DecisionTreeRegressor(random_state=42)


In [ ]:
# Fit the regressor to the training data
regressor.fit(X_train, y_train)


In [ ]:
# Evaluate the model

# Predictions
y_pred_train = regressor.predict(X_train)
y_pred_test = regressor.predict(X_test)

# Performance metrics
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("Decision Tree Performance (Unpruned)")
print(f"Train R²:  {train_r2:.3f}")
print(f"Test R²:   {test_r2:.3f}")
print(f"Train RMSE: {train_rmse:.2f}")
print(f"Test RMSE:  {test_rmse:.2f}")


In [ ]:
# Prune the model
regressor_pruned = DecisionTreeRegressor(
    max_depth=4,          # limits tree depth for interpretability
    min_samples_leaf=50,  # prevents very small, noisy leaf nodes
    random_state=42
)


## 4. Fit and plot final model.


In [ ]:
# Fit and plot final model
regressor_pruned.fit(X_train, y_train)

# Plot the final decision tree
plt.figure(figsize=(20, 10))

plot_tree(
    regressor_pruned,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title("Pruned Decision Tree for Loyalty Points")
plt.show()


**Justification of pruning strategy**

- A basic pre-pruning strategy was applied using max_depth=4 and min_samples_leaf=50. This approach was selected to balance predictive performance with interpretability and robustness. 
- Although the unpruned decision tree achieved extremely high performance (Train R² = 1.000; Test R² = 0.996), an unrestricted tree can become overly complex, unstable to small data changes, and difficult to communicate to non-technical stakeholders.
- Limiting tree depth reduces model complexity and prevents excessive branching, while enforcing a minimum leaf size ensures that each terminal node represents a meaningful customer segment rather than a small subset of observations.
- This produces a more usable model, consistent with the assignment objective to “grow and prune” a tree and interpret the resulting structure for decision-making.


**Insights and observations**

The decision tree model is highly useful as a decision-support and interpretability tool, rather than as a pure predictive model. The pruned tree reveals a clear and intuitive structure in loyalty point accumulation, with spending score identified as the primary driver of customer loyalty outcomes. This aligns with earlier regression results and provides additional value by translating statistical relationships into explicit, rule-based segments. The tree shows that customers with low spending scores consistently generate fewer loyalty points, while high-spending customers exhibit substantially higher loyalty engagement. Within these high-spending groups, remuneration acts as a secondary differentiator, refining segmentation and highlighting heterogeneity in loyalty behaviour. Age does not appear in the upper levels of the tree, confirming its limited relevance for loyalty decision-making.

The pruned structure enhances interpretability and robustness, making the model suitable for operational use. From a business perspective, the tree can inform transparent marketing rules, such as defining loyalty tiers, eligibility thresholds for rewards, and targeted incentives for high-potential customers. Future work could explore ensemble methods (e.g. random forests) to improve stability, incorporate additional behavioural variables such as purchase frequency or promotions, and analyse changes in loyalty behaviour over time. These steps would further strengthen evidence-based decision-making while retaining interpretability.


# 3. Customer segmentation with k-means


## 1. Load and explore the data


In [ ]:
# Import necessary libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import accuracy_score
from scipy.spatial.distance import cdist

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load the CSV file as df2
df2 = pd.read_csv("turtle_reviews_clean.csv")

# View DataFrame
df2.head(), df2.info()


In [ ]:
# Drop unnecessary columns (keep only clustering variables)
df3 = df2[["remuneration", "spending_score"]].copy()

# View DataFrame
df3.head(), df3.info()


In [ ]:
# Explore the data

# Basic overview
df3.head()

# Summary statistics
df3.describe()

# Check for missing values
df3.isna().sum()


In [ ]:
# Descriptive statistics
df3.describe()


## 2. Plot


In [ ]:
# Create a scatterplot with Seaborn
sns.scatterplot(
    data=df3,
    x="remuneration",
    y="spending_score",
    alpha=0.6
)

plt.title("Remuneration vs Spending Score (Seaborn Scatterplot)")
plt.xlabel("Remuneration (k£)")
plt.ylabel("Spending Score")
plt.show()


In [ ]:
# Create a pairplot with Seaborn
sns.pairplot(df3)
plt.suptitle("Pairplot of Remuneration and Spending Score", y=1.02)
plt.show()


## 3. Elbow and silhoutte methods


In [ ]:
# Determine the number of clusters: Elbow method

# Standardise the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df3)

# Compute inertia for different values of k
inertias = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Plot the Elbow curve
plt.figure(figsize=(8, 6))
plt.plot(list(k_range), inertias, marker="o", label="Inertia")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal Number of Clusters")
plt.legend()
plt.show()


In [ ]:
# Determine the number of clusters: Silhouette method

silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

# Plot Silhouette scores
plt.figure(figsize=(8, 6))
plt.plot(list(k_range), silhouette_scores, marker="o", label="Silhouette score")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette Method for Optimal Number of Clusters")
plt.legend()
plt.show()


## 4. Evaluate k-means model at different values of *k*


In [ ]:
# Evaluate k-means model at different values of k

candidate_ks = [3, 4, 5, 6, 7]  # includes your chosen k=5 and nearby alternatives

results = []

for k in candidate_ks:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    inertia = kmeans.inertia_
    sil = silhouette_score(X_scaled, labels)
    sizes = pd.Series(labels).value_counts().sort_index().to_dict()

    results.append({
        "k": k,
        "inertia": inertia,
        "silhouette": sil,
        "cluster_sizes": sizes
    })

results_df = pd.DataFrame(results).sort_values("k")
display(results_df[["k", "inertia", "silhouette"]])

print("\nCluster sizes per k:")
for _, row in results_df.iterrows():
    print(f"\nk = {int(row['k'])} | silhouette = {row['silhouette']:.3f} | inertia = {row['inertia']:.2f}")
    print(row["cluster_sizes"])


**Evaluation**

- Several values of k were evaluated using inertia, silhouette scores, and cluster sizes to balance statistical quality with practical interpretability.
- k = 3 produces very broad segments with a relatively low silhouette score (0.470). While simple, this solution groups heterogeneous customers together, limiting its usefulness for targeted marketing.
- k = 4 improves both inertia and silhouette score (0.511), indicating better separation than k = 3. However, one cluster remains disproportionately large (over 1,000 observations), suggesting that meaningful behavioural differences are still being merged.
- k = 5 provides the highest silhouette score (0.582) and a substantial reduction in inertia. Cluster sizes are well balanced, with no excessively small groups, making this solution both statistically robust and practically actionable. The segmentation captures meaningful variation in remuneration and spending behaviour without unnecessary complexity.
- k = 6 and k = 7 further reduce inertia but lead to declining silhouette scores and the emergence of smaller clusters. These solutions add complexity with limited additional interpretive or business value.


**Selection for final model**

Based on the combined evidence from the Elbow and Silhouette methods, k = 5 was selected as the optimal number of clusters. This solution achieves the best balance between cluster separation, stability, and interpretability, making it well suited for marketing segmentation and decision-making.


## 5. Fit final model and justify your choice


In [ ]:
# Apply the final k-means model
k_final = 5

kmeans_final = KMeans(
    n_clusters=k_final,
    random_state=42,
    n_init=10
)

# Fit the model and predict cluster labels
df3["cluster"] = kmeans_final.fit_predict(X_scaled)

# View the updated DataFrame
df3.head()


In [ ]:
# Number of observations per cluster
cluster_counts = df3["cluster"].value_counts().sort_index()
print("Number of observations per cluster:")
print(cluster_counts)


## 6. Plot and interpret the clusters


In [ ]:
# Visualising the clusters
plt.figure(figsize=(8, 6))
plt.scatter(
    df3["remuneration"],
    df3["spending_score"],
    c=df3["cluster"],
    cmap="viridis",
    alpha=0.6
)

plt.xlabel("Remuneration (k£)")
plt.ylabel("Spending Score")
plt.title("K-Means Clustering of Customers (k = 5)")
plt.colorbar(label="Cluster")
plt.show()


In [ ]:
# View the DataFrame
df3.head(), df3.tail(), df3.info()


**Cluster Observations**

Cluster 0 (356 customers): High Spending / Mid–High Remuneration
- Customers show strong engagement and above-average income.
- Action: Prioritise for loyalty retention, premium rewards, early access to products, and exclusive offers.

Cluster 1 (271 customers): Low–Mid Spending / Low Remuneration
- Lower engagement and more price-sensitive behaviour.
- Action: Use onboarding incentives, entry-level promotions, and value-focused bundles to increase engagement.

Cluster 2 (330 customers): Low Spending / High Remuneration
- High earning potential but currently under-engaged.
- Action: Target with personalised recommendations, convenience-focused offers, and nudges to increase spending frequency.

Cluster 3 (269 customers): High Spending / Low–Mid Remuneration
- Highly engaged despite lower income.
- Action: Reward loyalty with points multipliers, targeted discounts, and gamified incentives to maintain engagement without relying on price reductions.

Cluster 4 (774 customers): Mid Spending / Mid Remuneration
- Largest and most stable segment with moderate engagement.
- Action: Focus on progression strategies, such as tier-based rewards and targeted campaigns to move customers into higher-value segments.


K-means clustering was applied to remuneration and spending score to identify actionable customer segments for marketing decision-making. Both the Elbow and Silhouette methods indicated that five clusters provide the optimal balance between statistical performance and interpretability. The final solution produced reasonably balanced cluster sizes, ensuring that all segments are meaningful and suitable for targeted interventions.

The analysis shows that spending behaviour is the primary driver of customer segmentation, while remuneration refines differences within spending groups. High-spending customers consistently demonstrate higher engagement and should be prioritised for retention and premium loyalty initiatives. Customers with high remuneration but low spending represent clear growth opportunities, suggesting that targeted, personalised campaigns could unlock additional value. Lower-spending segments may benefit from onboarding support and value-focused incentives designed to increase engagement over time.

From a business perspective, this segmentation enables Turtle Games to move beyond broad demographic targeting toward behaviour-led, data-driven marketing strategies. Future work could enhance the model by incorporating additional behavioural variables such as purchase frequency, promotion responsiveness, or product preferences, and by validating cluster stability over time. These steps would further strengthen the organisation’s ability to design effective, evidence-based loyalty and marketing programmes.


# 4. Review text analysis (NLP)


## 1. Load and explore the data


In [ ]:
# Import all the necessary packages.
import pandas as pd
import numpy as np
import nltk 
import os 
import matplotlib.pyplot as plt

# nltk.download ('punkt').
# nltk.download ('stopwords').

from wordcloud import WordCloud
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from nltk.corpus import stopwords
from textblob import TextBlob
from scipy.stats import norm

# Import Counter.
from collections import Counter

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load the dataset for NLP
df4 = pd.read_csv("turtle_reviews.csv")

# View DataFrame
df4.head(), df4.info()


In [ ]:
# Explore data set.

# View first few rows
df4.head()

# Structure and data types
df4.info()

# Summary statistics (text length exploration)
df4["review_length"] = df4["review"].str.len()
df4["summary_length"] = df4["summary"].str.len()

df4[["review_length", "summary_length"]].describe()


In [ ]:
# Keep necessary columns for NLP
df4 = df4[["review", "summary"]].copy()

# View DataFrame
df4.head(), df4.info()


In [ ]:
# Determine if there are any missing values
df4.isna().sum()


## 2. Prepare the data for NLP
### 2a) Change to lower case and join the elements in each of the columns respectively (review and summary)


In [ ]:
# Review: Change all to lower case and join with a space
df4["review_clean"] = df4["review"].astype(str).str.lower()

# View result
df4[["review", "review_clean"]].head()


In [ ]:
# Summary: Change all to lower case and join with a space
df4["summary_clean"] = df4["summary"].astype(str).str.lower()

# View result
df4[["summary", "summary_clean"]].head()


### 2b) Replace punctuation in each of the columns respectively (review and summary)


In [ ]:
import string

# Replace all punctuation in review column
df4["review_clean"] = df4["review_clean"].str.translate(
    str.maketrans("", "", string.punctuation)
)

# View output
df4[["review", "review_clean"]].head()


In [ ]:
# Replace all punctuation in summary column
df4["summary_clean"] = df4["summary_clean"].str.translate(
    str.maketrans("", "", string.punctuation)
)

# View output
df4[["summary", "summary_clean"]].head()


### 2c) Drop duplicates in both columns


In [ ]:
# Drop duplicates in both cleaned columns
df4 = df4.drop_duplicates(subset=["review_clean", "summary_clean"])

# View DataFrame
df4.head(), df4.info()


## 3. Tokenise and create wordclouds


In [ ]:
# Create new DataFrame (copy DataFrame)
df4_copy = df4.copy()

# View DataFrame
df4_copy.head(), df4_copy.info()


In [ ]:
# Apply tokenisation
df4_copy["review_tokens"] = df4_copy["review_clean"].apply(word_tokenize)
df4_copy["summary_tokens"] = df4_copy["summary_clean"].apply(word_tokenize)

# View DataFrame
df4_copy[["review_tokens", "summary_tokens"]].head()


In [ ]:
# Review: Create a word cloud.

# Join cleaned review text
review_text = " ".join(df4_copy["review_clean"])

# Create the word cloud
wc_review = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate(review_text)


In [ ]:
# Review: Plot the WordCloud image.
plt.figure(figsize=(10, 5))
plt.imshow(wc_review, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud – Reviews (Raw Text)")
plt.show()


In [ ]:
# Summary: Create

# Join cleaned summary text
summary_text = " ".join(df4_copy["summary_clean"])

# Create the word cloud
wc_summary = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate(summary_text)


In [ ]:
# Summary: Plot the WordCloud image.
plt.figure(figsize=(10, 5))
plt.imshow(wc_summary, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud – Summaries (Raw Text)")
plt.show()


## 4. Frequency distribution and polarity
### 4a) Create frequency distribution


In [ ]:
# Flatten token lists (raw tokens)
review_words_raw = [word for tokens in df4_copy["review_tokens"] for word in tokens]
summary_words_raw = [word for tokens in df4_copy["summary_tokens"] for word in tokens]

# Frequency distributions (raw)
review_freq_raw = FreqDist(review_words_raw)
summary_freq_raw = FreqDist(summary_words_raw)

# View top 15 most common words (raw)
review_freq_raw.most_common(15), summary_freq_raw.most_common(15)


### 4b) Remove alphanumeric characters and stopwords


In [ ]:
# Keep alphabetic tokens only.
df4_copy["review_tokens_alpha"] = df4_copy["review_tokens"].apply(
    lambda tokens: [w for w in tokens if w.isalpha()]
)

df4_copy["summary_tokens_alpha"] = df4_copy["summary_tokens"].apply(
    lambda tokens: [w for w in tokens if w.isalpha()]
)

# View output
df4_copy[["review_tokens_alpha", "summary_tokens_alpha"]].head()


In [ ]:
# Remove all the stopwords

# Load stopwords
stop_words = set(stopwords.words("english"))

# Remove stopwords from alphabetic tokens
df4_copy["review_tokens_nostop"] = df4_copy["review_tokens_alpha"].apply(
    lambda tokens: [w for w in tokens if w not in stop_words]
)

df4_copy["summary_tokens_nostop"] = df4_copy["summary_tokens_alpha"].apply(
    lambda tokens: [w for w in tokens if w not in stop_words]
)

# View output
df4_copy[["review_tokens_nostop", "summary_tokens_nostop"]].head()


### 4c) Create wordcloud without stopwords


In [ ]:
# Create a wordcloud without stop words.

# Join cleaned tokens back into strings
review_text_nostop = " ".join(
    word for tokens in df4_copy["review_tokens_nostop"] for word in tokens
)

summary_text_nostop = " ".join(
    word for tokens in df4_copy["summary_tokens_nostop"] for word in tokens
)

# Review wordcloud (no stopwords)
wc_review_nostop = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate(review_text_nostop)

plt.figure(figsize=(10, 5))
plt.imshow(wc_review_nostop, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud – Reviews (No Stopwords)")
plt.show()

# Summary wordcloud (no stopwords)
wc_summary_nostop = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate(summary_text_nostop)

plt.figure(figsize=(10, 5))
plt.imshow(wc_summary_nostop, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud – Summaries (No Stopwords)")
plt.show()


### 4d) Identify 15 most common words and polarity


In [ ]:
# Determine the 15 most common words.

# Flatten cleaned token lists
review_words_clean = [
    word for tokens in df4_copy["review_tokens_nostop"] for word in tokens
]

summary_words_clean = [
    word for tokens in df4_copy["summary_tokens_nostop"] for word in tokens
]

# Frequency distributions
review_freq_clean = FreqDist(review_words_clean)
summary_freq_clean = FreqDist(summary_words_clean)

# 15 most common words
top15_reviews = review_freq_clean.most_common(15)
top15_summaries = summary_freq_clean.most_common(15)

top15_reviews, top15_summaries


## 5. Review polarity and sentiment: Plot histograms of polarity (use 15 bins) and sentiment scores for the respective columns.


In [ ]:
# Provided function.
def generate_polarity(comment):
    '''Extract polarity score (-1 to +1) for each comment'''
    return TextBlob(comment).sentiment[0]


In [ ]:
# Determine polarity of both columns
df4_copy["review_polarity"] = df4_copy["review_clean"].apply(generate_polarity)
df4_copy["summary_polarity"] = df4_copy["summary_clean"].apply(generate_polarity)

# View output
df4_copy[["review_clean", "review_polarity", "summary_clean", "summary_polarity"]].head()


In [ ]:
# Review: Create a histogram plot with bins = 15.

# Histogram of polarity
plt.figure(figsize=(8, 5))
plt.hist(df4_copy["review_polarity"], bins=15)
plt.xlabel("Polarity Score")
plt.ylabel("Frequency")
plt.title("Histogram of Review Polarity (15 bins)")
plt.show()


In [ ]:
# Histogram of sentiment score
plt.figure(figsize=(8, 5))
plt.hist(df4_copy["review_polarity"], bins=15)
plt.xlabel("Sentiment (Polarity) Score")
plt.ylabel("Frequency")
plt.title("Histogram of Review Sentiment Score (15 bins)")
plt.show()


**Analysis of 'Review' histograms**

- The polarity histograms for reviews and sentiment scores show identical distributions, as both measures are derived from the same polarity output of TextBlob.
- The distribution is positively skewed, with the highest frequency occurring at mildly positive polarity values, indicating general customer satisfaction with some variability in sentiment strength.


In [ ]:
# Summary: Create a histogram plot with bins = 15.
# Histogram of polarity
plt.figure(figsize=(8, 5))
plt.hist(df4_copy["summary_polarity"], bins=15)
plt.xlabel("Polarity Score")
plt.ylabel("Frequency")
plt.title("Histogram of Summary Polarity (15 bins)")
plt.show()


In [ ]:
# Histogram of sentiment score
plt.figure(figsize=(8, 5))
plt.hist(df4_copy["summary_polarity"], bins=15)
plt.xlabel("Sentiment (Polarity) Score")
plt.ylabel("Frequency")
plt.title("Histogram of Summary Sentiment Score (15 bins)")
plt.show()


**Analysis of 'Summary' histograms**

- Summary polarity shows a strong concentration at neutral values, reflecting the prevalence of short, descriptive summaries that contain limited sentiment-bearing language.
- When sentiment is expressed, it is predominantly positive, resulting in a positively skewed distribution with relatively few negative summaries.


## 6. Identify top 20 positive and negative reviews and summaries respectively


In [ ]:
# Top 20 negative reviews (lowest polarity scores)
top20_negative_reviews = (
    df4_copy
    .sort_values("review_polarity", ascending=True)
    .head(20)
)

# View output
top20_negative_reviews[["review", "review_polarity"]]


In [ ]:
# Top 20 negative summaries.

top20_negative_summaries = (
    df4_copy
    .sort_values("summary_polarity", ascending=True)
    .head(20)
)

# View output
top20_negative_summaries[["summary", "summary_polarity"]]

# View output.


In [ ]:
# Top 20 positive reviews.
top20_positive_reviews = (
    df4_copy
    .sort_values("review_polarity", ascending=False)
    .head(20)
)

# View output
top20_positive_reviews[["review", "review_polarity"]]


In [ ]:
# Top 20 positive summaries.
top20_positive_summaries = (
    df4_copy
    .sort_values("summary_polarity", ascending=False)
    .head(20)
)

# View output
top20_positive_summaries[["summary", "summary_polarity"]]


## 7. Additional Analysis


In [ ]:
#Comparing spread of polarity scores
df4_copy[["review_polarity", "summary_polarity"]].describe()


In [ ]:
#Finding extremes of polarity
review_extreme = (df4_copy["review_polarity"].abs() >= 0.75).mean()
summary_extreme = (df4_copy["summary_polarity"].abs() >= 0.75).mean()

review_extreme, summary_extreme


In [ ]:
#Concentration at neutral (0.00) polarity
(df4_copy["review_polarity"] == 0).mean(), (df4_copy["summary_polarity"] == 0).mean()


**Analysis**

- Quantitative analysis confirms that summaries exhibit more extreme sentiment than reviews.
- This is evidenced by a higher standard deviation in polarity scores (0.34 vs 0.26), a greater proportion of extreme sentiment values (11.8% vs 4.8%), and a much higher concentration of neutral scores.
- Reviews, by contrast, display a broader distribution of moderate polarity values, reflecting more nuanced and context-rich sentiment expression.
- These differences are consistent with the shorter, declarative nature of summaries compared with longer and more descriptive reviews.


Natural Language Processing of customer reviews and summaries indicates that overall sentiment toward Turtle Games products is strongly positive. Frequent terms such as game, fun, great, love, and five stars highlight enjoyment, perceived quality, replayability, and gift suitability as dominant drivers of customer satisfaction. Sentiment analysis confirms a positive skew in both reviews and summaries, with summaries exhibiting more extreme sentiment values due to their concise and declarative nature, while longer reviews provide more nuanced feedback.

Negative sentiment is comparatively limited but concentrated around actionable issues, particularly unclear or complex instructions, perceived poor value for money, and unmet expectations. These themes suggest opportunities for improvement in product documentation, onboarding, and expectation management rather than fundamental product flaws.

Further analysis could explore linking sentiment patterns to customer demographics or product categories to identify whether dissatisfaction is concentrated within specific segments. Future actions should include amplifying positive, experience-led themes in marketing communications, prioritising clarity and usability improvements in product design and instructions, and using summaries as a rapid sentiment monitoring tool to detect emerging issues early while leveraging full reviews for deeper qualitative insight.


# Key findings

- Spending behaviour was the strongest driver of loyalty-point accumulation, with remuneration providing additional explanatory value and age contributing less.
- A pruned decision tree highlighted non-linear behavioural thresholds that complemented the linear-regression results.
- K-means clustering identified five distinct customer groups based on spending and remuneration, creating a basis for differentiated marketing actions.
- Review and summary text showed broadly positive sentiment, while the most negative comments highlighted usability, value and instruction-related issues.
- The analysis demonstrates how several analytical methods can be combined, compared and translated into practical business recommendations rather than relying on a single model.
